<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/ASAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving training_set_rel3.tsv.zip to training_set_rel3.tsv (1).zip


In [ ]:
!unzip training_set_rel3.tsv.zip

Archive:  training_set_rel3.tsv.zip
replace training_set_rel3.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# Dataset Metadata Description
The dataset used in this study is the **Hewlett Foundation Automated Essay Scoring (AES) dataset**, originally released for the Kaggle Automated Student Assessment Prize (ASAP) competition. The dataset contains **12,976 student essays** written by students in grades **7 to 10**.

The dataset includes **28 variables** describing essay identifiers, essay prompts, essay text, and scores assigned by multiple human raters.

---

## Dataset Structure

Each row in the dataset represents **one student essay**, while each column represents an attribute associated with the essay.

The dataset dimensions are:

- **Number of essays:** 12,976  
- **Number of variables:** 28  

---

## Identification Variables

These columns uniquely identify each essay and its associated prompt.

- **essay_id**: A unique identifier for each essay.
- **essay_set**: Identifies which prompt the essay belongs to. The dataset contains **eight essay sets**, each corresponding to a different writing prompt.

---

## Essay Text

- **essay**: Contains the full textual response written by the student.

This column represents the **primary input data** used for automated essay scoring.

---

## Human Rater Scores

Each essay was evaluated by multiple human raters. The dataset includes the individual scores assigned by these raters.

- **rater1_domain1**
- **rater2_domain1**
- **rater3_domain1**

These columns represent scores assigned independently by different graders.

---

## Final Resolved Score

- **domain1_score**

This column represents the **final resolved score** determined after combining the individual rater scores. This value serves as the **ground truth label** used for training and evaluating automated essay scoring systems.

---

## Secondary Scoring Domain

Some essays include an additional scoring dimension.

- **rater1_domain2**
- **rater2_domain2**
- **domain2_score**

These scores are primarily associated with **essay set 2**, which evaluates essays across two scoring domains.

---

## Trait-Level Scores

Some essay sets include detailed trait-level scoring that evaluates specific aspects of writing quality.

Examples include:

- **rater1_trait1 – rater1_trait6**
- **rater2_trait1 – rater2_trait6**
- **rater3_trait1 – rater3_trait6**

These traits evaluate writing characteristics such as organization, grammar, coherence, and language usage.

---

## Missing Values in the Dataset

Certain columns contain **NaN (Not a Number)** values. This occurs because not all essay sets use the same scoring criteria.

For example:

- Some essay sets only contain **domain1 scores**.
- Essay set 2 includes **two scoring domains**.
- Essay sets 7–8 include **trait-level scoring**.

As a result, variables that are not applicable to certain essay sets appear as missing values.

---

## Key Variables Used in the Study

For the initial analysis and experimentation, the following variables are primarily used:

- **essay** → input text
- **domain1_score** → target score
- **essay_set** → prompt identifier

These variables provide the essential information required for automated essay scoring experiments.

In [ ]:
import pandas as pd

df = pd.read_csv('training_set_rel3.tsv', sep='\t', encoding='latin1')

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (12976, 28)

Columns:
['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']


The dataset contains 12,976 student essays and 28 associated variables describing essay identifiers, prompt identifiers, essay text, and scores assigned by human raters.
* Each essay is uniquely identified by an essay identifier and is associated with an essay set that corresponds to a specific writing prompt.
* Each essay was evaluated by multiple human raters. The dataset includes the individual scores assigned by these raters.
* The **domain1_score** column represents the resolved score derived from the individual rater evaluations and serves as the ground truth label for automated essay scoring experiments.
* Some essay sets include a secondary scoring dimension represented by domain2_score.

In [ ]:
import pandas as pd

df = pd.read_csv('training_set_rel3.tsv', sep='\t', encoding='latin1')
df.head()

In [ ]:
df['essay_set'].value_counts().sort_index() #Number of Essays in each essay set

In [ ]:
df.groupby('essay_set')['domain1_score'].agg(['min','max']) #score range for each essay set.

In [ ]:
df['essay_word_count'] = df['essay'].apply(lambda x: len(str(x).split()))
df['essay_word_count'].describe() #shortest essay, longest essay, average essay length in WORDS

**ESSAY SET**

In [ ]:
# Filter only essay set 1
set1 = df[df['essay_set'] == 1]

# check shape
set1.shape

In [ ]:
set1 = set1[['essay_id','essay','domain1_score']]

set1.head()

In [ ]:
set1['essay'] = set1['essay'].str.replace('\n',' ')
set1['essay'] = set1['essay'].str.strip()
set1.head() ##removes line breaks inside each essay.

In [ ]:
set1['domain1_score'].value_counts().sort_index()

In [ ]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1['score_category'] = set1['domain1_score'].apply(categorize)
set1[['domain1_score','score_category']].head()

In [ ]:
cal_low = set1[set1['score_category'] == 'Low'].sample(2, random_state=42)
cal_med = set1[set1['score_category'] == 'Medium'].sample(2, random_state=42)
cal_high = set1[set1['score_category'] == 'High'].sample(2, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).reset_index(drop=True)
calibration

In [ ]:
demo = set1.drop(calibration.index, errors='ignore').sample(5, random_state=42).reset_index(drop=True)

demo

# **TRIAL ONE (Calibration+ essays for each essay)**

In [ ]:
def format_calibration_examples(calibration_df):
    examples = ""
    for i, row in calibration_df.iterrows():
        examples += f"""
Calibration Example {i+1}

Essay:
{row['essay']}

Human Score: {row['domain1_score']}
Category: {row['score_category']}
"""
    return examples

calibration_text = format_calibration_examples(calibration)
print(calibration_text)


Calibration Example 1335

Essay:
would you rather read six books to get that one recipie, or would you rather look it up in less than ten minutes? I think computers have an positive effect on people you can learn about places you want you need in minutes and you can talk to your friends inline Computers have a good effect on people because, you can learn about places you want to wish this @DATE1. For example, @PERSON5 wanted to wish @LOCATION2 this @DATE1 but when he looked online he saw how @LOCATION2 just had an earthquake so he changed his plans this @DATE1. @CAPS1, @PERSON2 said "I wanted to go to @LOCATION3 this @DATE1 but I know about it, so I searched on google and figure thatb this @DATE1 I will go to @LOCATION3" that is one positive thing computers effect on people because you can learn about places you want to visit in the @DATE1. Computers have a positive effect on people because you can @CAPS1 find what you need in minutes. For example. @PERSON3 was looking for a cake reci

In [ ]:
demo_essay = demo.iloc[0]['essay']
print(demo_essay)

Hello, I decided to write in to the paper on the topic of computers, and how they benefit society. Comuters are powerful machines, and can be used completing otherwise menial and boring tasks. They benefit people in many ways, like they are very useful in buissness, helpful in education, and connect friends and family together. Computers are uselful in buissness and are an extremely important tool in a common workplace today, and make work get done much more efficiently than if they didn't have computers. One reason Computers are so useful in the workplace is because computers can storee far more data that can be put on paper. Also, they can do jobs and processes very quickly, which is important in the workplace. They can also be used for sending important information quickly and without much trouble. These are all valid reasons comuters are beneficial in buissness, and help things get done. Computers are important today's schools and educational programs becasue they can help students

In [ ]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [ ]:
full_prompt = f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{demo_essay}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

print(full_prompt)


You are an expert essay grader evaluating student essays.


Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.


Calibration Examples:

Calibration Example 1335

Essay:
would you rather read six books to get that one recipie, or would you rather look it up in less than ten minutes? I think computers have an positive effect on people you can learn about places you want you need in minutes and you can talk to your friends inline Computers have

In [ ]:
def build_prompt(essay_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.
4. Base your explanation only on details actually present in the essay.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [ ]:
def build_prompt(essay_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.
4. Base your explanation only on details actually present in the essay.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [ ]:
demo['full_prompt'] = demo['essay'].apply(build_prompt)

In [ ]:
print(demo.loc[0, 'full_prompt'])

In [ ]:
print(demo.loc[1, 'full_prompt'])

In [ ]:
print(demo.loc[2, 'full_prompt'])


In [ ]:
print(demo.loc[3, 'full_prompt'])


In [ ]:
print(demo.loc[4, 'full_prompt'])


---
## TRIAL TWO (Calibration once in the beginning)


In [ ]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

def format_calibration_examples(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text = format_calibration_examples(calibration)

calibration_prompt = f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Use the following calibration examples to learn the expected scoring standard.

{calibration_text}

Important instructions:
- Use the rubric and calibration examples as the scoring standard for future essays.
- When grading future essays, predict one final score from 2 to 12.
- Also assign a category:
  Low = score 2–5
  Medium = score 6–8
  High = score 9–12
- Base your reasoning only on what is actually written in the essay.

Reply only with:
Calibration received.
"""

print(calibration_prompt)

In [ ]:
def build_test_prompt(essay_id, essay_text):
    return f"""
Now evaluate the following essay.

Essay ID: {essay_id}

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning briefly.
4. Base your explanation only on details actually present in the essay.

Return your answer exactly in this format:

Essay ID: {essay_id}
Final Score: <number>
Category: <Low/Medium/High>
Reasoning: <your explanation>
"""

**table with essays and scores to compare them later to ai:**

In [ ]:
test_df = test_df.reset_index(drop=True).copy()

test_df["test_prompt"] = test_df.apply(
    lambda row: build_test_prompt(row["essay_id"], row["essay"]),
    axis=1
)

test_df[["essay_id", "domain1_score", "score_category"]]

In [ ]:
print(test_df.loc[0, "test_prompt"])

In [ ]:
print(test_df.loc[1, "test_prompt"])

In [ ]:
print(test_df.loc[2, "test_prompt"])

In [ ]:
print(test_df.loc[3, "test_prompt"])

In [ ]:
print(test_df.loc[4, "test_prompt"])

# **FINAL**

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("training_set_rel3.tsv", sep="\t", encoding="latin1")

# Keep essay set 1
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]].copy()

# Clean essay text
set1["essay"] = set1["essay"].astype(str).str.replace("\n", " ", regex=False).str.strip()

# Create score category
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)

# Calibration set
cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=42)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=42)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).copy()

# Remove calibration essays from testing pool
test_pool = set1[~set1["essay_id"].isin(calibration["essay_id"])].copy()

# Test essays
test_df = test_pool.sample(5, random_state=42).reset_index(drop=True).copy()

print("Calibration IDs:", calibration["essay_id"].tolist())
print("Test IDs:", test_df["essay_id"].tolist())

# Safety check
overlap = set(calibration["essay_id"]).intersection(set(test_df["essay_id"]))
print("Overlap:", overlap)

In [ ]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

def format_calibration_examples(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text = format_calibration_examples(calibration)

calibration_prompt = f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Use the following calibration examples to learn the expected scoring standard.

{calibration_text}

Important instructions:
- Use the rubric and calibration examples as the scoring standard for future essays.
- When grading future essays, predict one final score from 2 to 12.
- Also assign a category:
  Low = score 2–5
  Medium = score 6–8
  High = score 9–12
- Base your reasoning only on what is actually written in the essay.

Reply only with:
Calibration received.
"""

print(calibration_prompt)

In [ ]:
def build_test_prompt(essay_id, essay_text):
    return f"""
Now evaluate the following essay.

Essay ID: {essay_id}

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning briefly.
4. Base your explanation only on details actually present in the essay.

Return your answer exactly in this format:

Essay ID: {essay_id}
Final Score: <number>
Category: <Low/Medium/High>
Reasoning: <your explanation>
"""

tracking table

In [ ]:
tracking_df = test_df[[
    "essay_id",
    "essay",
    "domain1_score",
    "score_category"
]].copy()

tracking_df = tracking_df.rename(columns={
    "domain1_score": "human_score",
    "score_category": "human_category"
})

tracking_df["prompt"] = tracking_df.apply(
    lambda row: build_test_prompt(row["essay_id"], row["essay"]),
    axis=1
)

tracking_df["ai_raw_response"] = ""
tracking_df["ai_score"] = None
tracking_df["ai_category"] = ""

tracking_df[["essay_id", "human_score", "human_category"]]

In [ ]:
print(tracking_df.loc[0, "prompt"])
print(tracking_df.loc[1, "prompt"])
print(tracking_df.loc[2, "prompt"])
print(tracking_df.loc[3, "prompt"])
print(tracking_df.loc[4, "prompt"])

In [ ]:
# ============================
# 6) CREATE MASTER TRACKING DF
# ============================

tracking_df = test_df[[
    "essay_id",
    "essay",
    "domain1_score",
    "score_category"
]].copy()

tracking_df = tracking_df.rename(columns={
    "domain1_score": "human_score",
    "score_category": "human_category"
})

tracking_df["prompt"] = tracking_df.apply(
    lambda row: build_test_prompt(row["essay_id"], row["essay"]),
    axis=1
)

# Columns to paste Ollama outputs into later
tracking_df["ai_raw_response"] = ""
tracking_df["ai_score"] = pd.NA
tracking_df["ai_category"] = pd.NA
tracking_df["score_diff"] = pd.NA
tracking_df["abs_error"] = pd.NA

print("\n===== TRACKING TABLE =====")
print(tracking_df[["essay_id", "human_score", "human_category"]])


# ========================================
# 7) DISPLAY THE TEST PROMPTS ONE BY ONE
# ========================================

print("\n===== TEST PROMPT 0 =====")
print(tracking_df.loc[0, "prompt"])

print("\n===== TEST PROMPT 1 =====")
print(tracking_df.loc[1, "prompt"])

print("\n===== TEST PROMPT 2 =====")
print(tracking_df.loc[2, "prompt"])

print("\n===== TEST PROMPT 3 =====")
print(tracking_df.loc[3, "prompt"])

print("\n===== TEST PROMPT 4 =====")
print(tracking_df.loc[4, "prompt"])


# ======================================================
# 8) AFTER TESTING IN OLLAMA, PASTE RAW RESPONSES BELOW
# ======================================================
# Replace the text inside the triple quotes with the actual Ollama outputs.
# Keep the same row numbers aligned with the prompt you tested.

tracking_df.loc[0, "ai_raw_response"] = "Essay ID: 66 Final Score: 8 Category: Medium Reasoning: The essay presents a clear and coherent argument on the benefits of using computers in everyday life. The writer provides specific examples, such as communicating with people across distances, conducting research, and completing homework tasks more efficiently. The essay also acknowledges the emotional impact of technology, citing an example of reconnection between two family members. While the writing is not exceptionally polished or sophisticated, the ideas are well-organized and easy to follow. Overall, the essay demonstrates a solid understanding of the topic and presents a persuasive case for the advantages of computer use."

tracking_df.loc[1, "ai_raw_response"] = "Essay ID: 946 Final Score: 10 Category: High Reasoning: The essay presents a well-structured and persuasive argument against the excessive use of computers. The writer uses personal anecdotes to illustrate the negative consequences of computer addiction, including decreased physical activity, neglect of family relationships, and a lack of enjoyment in life. The examples provided are specific, relatable, and effectively convey the writer's point. The essay also demonstrates an understanding of the psychological impact of computer use, highlighting the importance of human connection and experiences over screen time. Overall, the writing is clear, concise, and well-organized, making it a strong persuasive piece that effectively argues against the misuse of computers."

tracking_df.loc[2, "ai_raw_response"] = """
Essay ID: 837
Final Score: 6
Category: Low
Reasoning: The essay lacks a clear and cohesive structure, with the writer jumping abruptly between different topics and ideas. While the writer attempts to highlight the benefits of computers, such as language learning and video chatting, these points are not fully developed or supported by evidence. Additionally, the writing is informal and lacks polish, with frequent use of colloquialisms ("Dear @ORGANIZATION1") and poor sentence structure ("And now you are probably asking..."). The essay also includes unnecessary details, such as the cost of Skype calls, which do not contribute to the overall argument. Overall, the essay requires significant improvement in terms of organization, clarity, and writing quality.
"""


tracking_df.loc[3, "ai_raw_response"] = "Essay ID: 1710 Final Score: 8 Category: Medium Reasoning: The essay presents a clear and organized argument on the positive effects of computers on people. The writer provides three main reasons why computers have a good affect on people, including improved typing skills, educational value, and assistance with various tasks. While the writing is not exceptional in terms of grammar, syntax, or vocabulary, it is generally easy to follow and understand. The essay also demonstrates some thoughtfulness, such as highlighting the potential benefits for individuals with hearing impairments. However, the writing could benefit from more development and support of ideas, as well as more attention to sentence structure and clarity. Overall, the essay shows promise but requires refinement in certain areas."


tracking_df.loc[4, "ai_raw_response"] = "Essay ID: 804 Final Score: 9 Category: High Reasoning: The essay presents a strong and persuasive argument against the negative effects of computers on people. The writer provides three main points highlighting the problems caused by excessive computer use, including health risks, social isolation, and online safety concerns. The writing is clear, concise, and easy to follow, with specific examples and anecdotes that support the author's claims. Additionally, the essay demonstrates a good understanding of the topic and presents a logical argument against the misuse of computers. While there may be some minor errors in grammar or syntax, they do not detract from the overall impact of the essay. Overall, this is a well-written and effective persuasive piece that presents a compelling case against the negative effects of computer addiction."


# ============================================
# 9) EXTRACT AI SCORE AND CATEGORY AUTOMATICALLY
# ============================================
import re
def extract_score(text):
    match = re.search(r"Final Score:\s*(\d+)", str(text), re.IGNORECASE)
    return int(match.group(1)) if match else pd.NA

def extract_category(text):
    match = re.search(r"Category:\s*(Low|Medium|High)", str(text), re.IGNORECASE)
    return match.group(1).capitalize() if match else pd.NA

tracking_df["ai_score"] = tracking_df["ai_raw_response"].apply(extract_score)
tracking_df["ai_category"] = tracking_df["ai_raw_response"].apply(extract_category)

# Convert scores to numeric safely
tracking_df["human_score"] = pd.to_numeric(tracking_df["human_score"], errors="coerce")
tracking_df["ai_score"] = pd.to_numeric(tracking_df["ai_score"], errors="coerce")

tracking_df["score_diff"] = tracking_df["ai_score"] - tracking_df["human_score"]
tracking_df["abs_error"] = tracking_df["score_diff"].abs()

print("\n===== COMPARISON TABLE =====")
print(tracking_df[[
    "essay_id",
    "human_score",
    "ai_score",
    "human_category",
    "ai_category",
    "score_diff",
    "abs_error"
]])


# ============================
# 10) COMPUTE QWK AND PCC
# ============================
from sklearn.metrics import cohen_kappa_score
from scipy.stats import pearsonr
valid_df = tracking_df.dropna(subset=["human_score", "ai_score"]).copy()

if len(valid_df) >= 2:
    qwk = cohen_kappa_score(
        valid_df["human_score"],
        valid_df["ai_score"],
        weights="quadratic"
    )

    pcc, _ = pearsonr(
        valid_df["human_score"],
        valid_df["ai_score"]
    )

    print("\n===== METRICS =====")
    print("QWK:", qwk)
    print("PCC:", pcc)
else:
    print("\nNot enough valid AI scores yet to compute QWK and PCC.")


# ==========================================
# 11) OPTIONAL: SAVE RESULTS TO CSV
# ==========================================

tracking_df.to_csv("essay_scoring_results.csv", index=False)
print("\nResults saved to essay_scoring_results.csv")

# **bigger calibration and test set**
(with enhanced rubric prompt)

5 Low, 5 Medium, 5 High = 15 essays ✅

In [ ]:
# Recreate calibration with larger size
cal_low = set1[set1["score_category"] == "Low"].sample(5, random_state=42)
cal_med = set1[set1["score_category"] == "Medium"].sample(5, random_state=42)
cal_high = set1[set1["score_category"] == "High"].sample(5, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).copy()

# Recreate test pool
test_pool = set1[~set1["essay_id"].isin(calibration["essay_id"])].copy()

# Pick test size
n_test = 15   # start with 10 or 15 since you're doing Ollama manually
test_df = test_pool.sample(n_test, random_state=42).reset_index(drop=True).copy()

print("Calibration size:", len(calibration))
print("Test size:", len(test_df))
print("Calibration IDs:", calibration["essay_id"].tolist()[:10], "...")
print("Test IDs:", test_df["essay_id"].tolist()[:10], "...")
print("Overlap:", set(calibration["essay_id"]).intersection(set(test_df["essay_id"])))

Calibration size: 15
Test size: 15
Calibration IDs: [1339, 686, 1592, 772, 1572, 951, 1096, 995, 1212, 1536] ...
Test IDs: [982, 277, 415, 970, 523, 1467, 1094, 348, 1059, 1197] ...
Overlap: set()


In [38]:
def format_calibration_examples(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text = format_calibration_examples(calibration)

calibration_prompt = f"""
You are an expert essay grader evaluating Grade 8 student essays.

========================
STUDENT TASK (FULL PROMPT)
========================

More and more people use computers, but not everyone agrees that this benefits society.
Those who support advances in technology believe that computers have a positive effect on people.
They teach hand-eye coordination, give people the ability to learn about faraway places and people,
and even allow people to talk online with other people.

Others have different ideas. Some experts are concerned that people are spending too much time
on their computers and less time exercising, enjoying nature, and interacting with family and friends.

Write a letter to your local newspaper in which you state your opinion on the effects computers have on people.
Persuade the readers to agree with you.

========================
SCORING SYSTEM
========================

- Each essay is scored by TWO human raters.
- Each rater assigns a score from 1 to 6.
- The final score is the SUM of both scores → range: 2 to 12.

ADJUDICATION RULE:
- If the two scores are adjacent → final score = sum
- If the two scores are NOT adjacent → the final score is determined by an expert scorer

Your task is to predict the FINAL resolved score (2–12).

========================
IMPORTANT DATA CHARACTERISTICS
========================

- Essays are written by students and may contain spelling, grammar, and punctuation errors.
- These errors should be preserved and should NOT overly influence scoring unless they affect clarity.
- Focus primarily on argument quality, reasoning, and structure.

========================
RUBRIC DIMENSIONS
========================

Evaluate essays based on:

1. Development of ideas (depth, clarity, elaboration)
2. Organization (logical structure and flow)
3. Clarity and fluency (readability and coherence)
4. Use of supporting details (specific vs general)
5. Awareness of audience (persuasive tone and purpose)

========================
RUBRIC LEVELS (1–6)
========================

Score Point 1:
- Undeveloped response
- Very minimal support
- Few or vague details
- Awkward and fragmented
- No awareness of audience
- Difficult to understand

Score Point 2:
- Underdeveloped response
- General or list-like ideas
- Weak reasoning
- Little or no organization
- Simplistic or unclear
- Limited awareness of audience

Score Point 3:
- Minimally developed response
- Some reasoning but limited elaboration
- More general than specific details
- Basic organization
- Weak transitions
- Some awareness of audience

Score Point 4:
- Somewhat developed response
- Adequate argument
- Mix of general and specific details
- Satisfactory organization
- Some transitions
- Adequate awareness of audience

Score Point 5:
- Developed response
- Clear position and persuasive support
- Mostly specific and relevant details
- Strong organization
- Good use of transitions
- Consistent awareness of audience

Score Point 6:
- Well-developed response
- Clear, thoughtful, and persuasive argument
- Fully elaborated ideas with specific details
- Strong and logical organization
- Fluent writing with effective transitions
- Strong awareness of audience

Use these levels internally to simulate how two human raters would score.

========================
CALIBRATION EXAMPLES
========================

{calibration_text}

========================
CRITICAL SCORING BEHAVIOR
========================

- Use calibration examples as strict reference points.
- Do NOT over-score essays with weak development.
- Do NOT inflate scores based only on essay length.
- Be strict when support is vague or repetitive.
- Allow minor grammar/spelling errors if ideas are strong.
- Ensure scores remain consistent with calibration patterns.

========================
GOAL
========================

Your goal is to simulate human grading behavior as closely as possible.

========================
OUTPUT
========================

Reply only with:
Calibration received.
"""

In [39]:
def build_test_prompt(essay_id, essay_text):
    return f"""
Now evaluate the following essay.

Essay ID: {essay_id}

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning briefly.
4. Base your explanation only on details actually present in the essay.

Return your answer exactly in this format:

Essay ID: {essay_id}
Final Score: <number>
Category: <Low/Medium/High>
Reasoning: <your explanation>
"""

tracking_df = test_df[["essay_id", "essay", "domain1_score", "score_category"]].copy()

tracking_df = tracking_df.rename(columns={
    "domain1_score": "human_score",
    "score_category": "human_category"
})

tracking_df["prompt"] = tracking_df.apply(
    lambda row: build_test_prompt(row["essay_id"], row["essay"]),
    axis=1
)

tracking_df["ai_raw_response"] = ""
tracking_df["ai_score"] = pd.NA
tracking_df["ai_category"] = pd.NA
tracking_df["score_diff"] = pd.NA
tracking_df["abs_error"] = pd.NA

tracking_df[["essay_id", "human_score", "human_category"]].head()

,essay_id,human_score,human_category
0,982,8,Medium
1,277,9,High
2,415,7,Medium
3,970,11,High
4,523,10,High
